# Gabarito Exercício:

### No código a seguir faça uma compensação usando algum dos quatro tipos de compensação contínua. Crie uma biblioteca com os seguintes tipos de compensação: P, PI, PD e PID. E os use para melhorar o sistema presente no código:

In [ ]:
%pip install control

In [ ]:
# Dicionário dos controladores
# Os valores servem apenas como valores iniciais dos sliders

controladores = {
    "P": {
        "parametros": {"Kp": 1.0}
    },
    "PI": {
        "parametros": {"Kp": 1.0, "Ki": 1.0}
    },
    "PD": {
        "parametros": {"Kp": 1.0, "Kd": 1.0}
    },
    "PID": {
        "parametros": {"Kp": 1.0, "Ki": 1.0, "Kd": 1.0}
    }
}

print("Tipos disponíveis:")
for nome in controladores:
    print(f"- {nome}")

tipo = input("\nEscolha o controlador (P, PI, PD ou PID): ").upper()

if tipo not in controladores:
    raise ValueError("Tipo de controlador inválido!")

print(f"\nControlador selecionado: {tipo}")

In [ ]:
import control as ct
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider

# ==========================================================
# Sistema a ser controlado
# ==========================================================

num_sistema = [1]
den_sistema = [1, 4, 3]

G = ct.TransferFunction(num_sistema, den_sistema)

# Sistema sem compensação
T_sem = ct.feedback(G)

# Vetor de tempo
t = np.linspace(0, 10, 1000)

# ==========================================================
# Função de atualização
# ==========================================================

def atualizar(**ganhos):

    Kp = ganhos.get("Kp", 0)
    Ki = ganhos.get("Ki", 0)
    Kd = ganhos.get("Kd", 0)

    # Construção automática do controlador
    if tipo == "P":

        C = ct.TransferFunction(
            [Kp],
            [1]
        )

    elif tipo == "PI":

        C = ct.TransferFunction(
            [Kp, Ki],
            [1, 0]
        )

    elif tipo == "PD":

        C = ct.TransferFunction(
            [Kd, Kp],
            [1]
        )

    elif tipo == "PID":

        C = ct.TransferFunction(
            [Kd, Kp, Ki],
            [1, 0]
        )

    # Sistema compensado
    T_com = ct.feedback(C * G)

    # Respostas ao degrau
    t1, y1 = ct.step_response(T_sem, t)
    t2, y2 = ct.step_response(T_com, t)

    # Limpa saída anterior
    plt.close('all')

    # ======================================================
    # Gráfico
    # ======================================================

    plt.figure(figsize=(10, 6))

    plt.plot(
        t1,
        y1,
        '--',
        linewidth=2,
        label='Sem compensação'
    )

    plt.plot(
        t2,
        y2,
        linewidth=2,
        label='Com compensação'
    )

    plt.title(f'Resposta ao Degrau - Controlador {tipo}')
    plt.xlabel('Tempo (s)')
    plt.ylabel('Amplitude')
    plt.grid(True)
    plt.legend()

    plt.show()

    # ======================================================
    # Informações
    # ======================================================

    print("Sistema G(s):")
    print(G)

    print("\nControlador C(s):")
    print(C)

    print("\nSistema compensado:")
    print(T_com)

# ==========================================================
# Criação automática dos sliders
# ==========================================================

sliders = {}

for parametro, valor_inicial in controladores[tipo]["parametros"].items():

    sliders[parametro] = FloatSlider(
        value=valor_inicial,
        min=0,
        max=20,
        step=0.1,
        description=parametro,
        continuous_update=True
    )

# ==========================================================
# Interface interativa
# ==========================================================

interact(
    atualizar,
    **sliders
);